# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.gaussian_process.kernels import (
    RBF, WhiteKernel, ConstantKernel as C, DotProduct
)
from joblib import dump, load
import matplotlib.pyplot as plt
import mplcursors
import plotly.express as px
import os
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", message="X has feature names, but GaussianProcessRegressor was fitted without feature names")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
file_path = '../chemical-featurization/full_descriptors/repeat_rdkit_physchem.xlsx'
model_output = 'gisaxs_fwhm'
# import data frame for pre-processing
df = pd.read_excel(file_path)

In [ ]:
# Drop rows with any NaN values
df_clean = df.dropna()

# Assign input variables and target variable
inputs = df_clean.iloc[:, 1:-2].copy()
target = df_clean[f'{model_output}']

# GPR training

In [ ]:
%%time

X_train, X_test, y_train, y_test = train_test_split(
    inputs, target, test_size=0.2, random_state=10
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

N_PCA_COMPONENTS = 49   # <-- Too computationaly expensive to run on all dimensions, use PCA reduction

pca = PCA(n_components=N_PCA_COMPONENTS)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"Original dims: {X_train.shape[1]}")
print(f"PCA dims:      {X_train_pca.shape[1]}")

n_features = X_train_pca.shape[1]

kernel = (

    C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0)
    + C(1.0, (1e-2, 1e2)) * RBF(
        length_scale=np.ones(n_features),
        length_scale_bounds=(1e-2, 50.0)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e-1)
    )
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    alpha=1e-10,
    random_state=42
)

gpr.fit(X_train_pca, y_train)

y_pred, y_std = gpr.predict(X_test_pca, return_std=True)
test_r2 = r2_score(y_test, y_pred)

print("\nLearned Kernel:")
print(gpr.kernel_)
print(f"\nTest R²: {test_r2}")